In [4]:
import numpy as np
import shapiq
from sklearn.ensemble import RandomForestRegressor
from shapiq.approximator import ProxySHAP

# 1. Daten laden (California Housing)
X, y = shapiq.load_california_housing(to_numpy=True)
n_features = X.shape[1]

print(f"Anzahl der Features im Game: {n_features}") # Sollte 8 sein

# 2. Modell trainieren
model = RandomForestRegressor(max_depth=4, random_state=42)
model.fit(X, y)

# 3. Vorbereitungen für die Game Function
# Wir wollen die Vorhersage für das allererste Haus (Index 0) erklären
x_explain = X[0] 
# Durchschnittswerte des Datensatzes als Referenzwerte berechnen
reference_values = X.mean(axis=0)

# 4. Deine Game Function einfügen
# (Diese ersetzt nun die Rolle des MarginalImputers)
def game_function(coalitions: np.ndarray) -> np.ndarray:
    """
    Value function for explaining x_explain.
    """
    batch_size = coalitions.shape[0]
    
    # 1. Wandle die 0/1 Matrix explizit in True/False (Boolean) um
    mask = coalitions.astype(bool)
    
    # 2. Erstelle die Matrix mit Durchschnittswerten (reference_values)
    inputs = np.tile(reference_values, (batch_size, 1))
    
    # 3. Nutze die Boolesche Maske, um die echten Werte einzufügen
    inputs[mask] = np.tile(x_explain, (batch_size, 1))[mask]
    
    # 4. Vorhersage machen und zurückgeben
    return model.predict(inputs)

# 5. Den ProxySHAP Approximator nutzen
# (Dieser ersetzt nun den ExactComputer)
approx = ProxySHAP(n=n_features, random_state=42)

# Shapley Values mit einem Budget von z.B. 2048 berechnen
proxy_sv = approx.approximate(budget=2048, game=game_function)

print("\nVorhersage des Modells für X[0]:", model.predict(X[[0]])[0])
print("\nProxySHAP Shapley Values für das Housing Dataset (X[0]):")
print(proxy_sv)

Anzahl der Features im Game: 8

Vorhersage des Modells für X[0]: 4.5510733911328165

ProxySHAP Shapley Values für das Housing Dataset (X[0]):
InteractionValues(
    index=SII, max_order=2, min_order=0, estimated=True, estimation_budget=2048,
    n_players=8, baseline_value=1.74743772062558,
    Top 10 interactions:
        (0,): 2.7952060069055293
        (): 1.74743772062558
        (0, 1): 0.008383517049739133
        (5,): 0.004238129449006396
        (1,): 0.004191758523747932
        (1, 2): 0.0
        (1, 6): 0.0
        (1, 7): 0.0
        (2,): -3.3881317890172014e-21
        (0, 5): -0.008476258899608855
)


/Users/zheshi/Documents/GitHub/shapiq/src/shapiq/approximator/proxy/proxyshap.py:244: UserWarning: Not all budget is required due to the border-trick.
  self._sampler.sample(budget)
/Users/zheshi/Documents/GitHub/shapiq/src/shapiq/approximator/proxy/proxyshap.py:116: UserWarning: Not all budget is required due to the border-trick.
  self._sampler.sample(budget)
